In [4]:
aca_gemma4_31b_it_a100_fqdn = ! terraform output -raw aca_gemma4_31b_it_a100_fqdn
aca_gemma4_31b_it_a100_fqdn = aca_gemma4_31b_it_a100_fqdn.n
print("LLM Endpoint:", aca_gemma4_31b_it_a100_fqdn)

sessionpool_management_endpoint = ! terraform output -raw sessionpool_management_endpoint
sessionpool_management_endpoint = sessionpool_management_endpoint.n
print("Session Pool Management Endpoint:", sessionpool_management_endpoint)

mcp_sessionpool_endpoint = ! terraform output -raw mcp_sessionpool_endpoint
mcp_sessionpool_endpoint = mcp_sessionpool_endpoint.n
print("MCP Session Pool Endpoint:", mcp_sessionpool_endpoint)

LLM Endpoint: gemma-4-31b-it-a100.gentlemushroom-793350b5.swedencentral.azurecontainerapps.io
Session Pool Management Endpoint: https://swedencentral.dynamicsessions.io/subscriptions/dcef7009-6b94-4382-afdc-17eb160d709a/resourceGroups/rg-aca-gpu-nvidia-555/sessionPools/acasessionpool
MCP Session Pool Endpoint: https://swedencentral.dynamicsessions.io/subscriptions/dcef7009-6b94-4382-afdc-17eb160d709a/resourceGroups/rg-aca-gpu-nvidia-555/sessionPools/acasessionpool/mcp


In [ ]:
%pip install langchain langchain-openai langchain-mcp-adapters langchain-azure-dynamic-sessions

In [6]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

model = ChatOpenAI(
    base_url=f"http://{aca_gemma4_31b_it_a100_fqdn}/v1",
    api_key="EMPTY",
    model="google/gemma-4-31B-it",
    streaming=True,
    max_completion_tokens= 512
)

response = model.stream([HumanMessage(content="Tell me about yourself.")])

for chunk in response:
    print(chunk.content, end="", flush=True)

I am a large language model, trained by Google. 

Think of me as a versatile, AI-powered collaborator. I don’t have a physical form, a personal history, or feelings, but I have been trained on a massive amount of text data, which allows me to process information and generate human-like text across a huge variety of tasks.

Here is a breakdown of what I can do and how I work:

### 🛠️ What I can do for you
*   **Writing & Creativity:** I can write emails, essays, poems, scripts, blog posts, and stories. I can also help you brainstorm ideas or refine your own writing.
*   **Learning & Information:** I can summarize long articles, explain complex scientific concepts in simple terms, or give you a crash course on a historical event.
*   **Problem Solving & Logic:** I can help with coding (in many different languages), solve math problems, and help you organize your thoughts or plan a project.
*   **Analysis:** I can take a piece of text and extract the key points, change the tone, or transl

In [7]:
from langchain.agents import create_agent
from langchain_azure_dynamic_sessions.tools import SessionsPythonREPLTool
from azure.identity import AzureCliCredential

credential = AzureCliCredential()

def access_token_provider():
    token = credential.get_token("https://dynamicsessions.io/.default")
    return token.token

# get the management endpoint from the session pool in the Azure portal
toolPythonSession = SessionsPythonREPLTool(
    pool_management_endpoint=sessionpool_management_endpoint,
    access_token_provider=access_token_provider,
)

agent = create_agent(model=model, tools=[toolPythonSession])

async for step in agent.astream(
    {"messages": [{"role": "user", "content": "What is the current time in Tunisia and France ?"}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What is the current time in Tunisia and France ?


================================== Ai Message ==================================
Tool Calls:
  Python_REPL (chatcmpl-tool-9178ea170e4eb1a7)
 Call ID: chatcmpl-tool-9178ea170e4eb1a7
  Args:
    python_code: from datetime import datetime
import pytz

tunisia_tz = pytz.timezone('Africa/Tunis')
france_tz = pytz.timezone('Europe/Paris')

tunisia_time = datetime.now(tunisia_tz).strftime('%H:%M:%S')
france_time = datetime.now(france_tz).strftime('%H:%M:%S')

print(f"Tunisia: {tunisia_time}")
print(f"France: {france_time}")
================================= Tool Message =================================
Name: Python_REPL

{
  "result": "",
  "stdout": "Tunisia: 05:43:19\nFrance: 06:43:19\n",
  "stderr": ""
}
================================== Ai Message ==================================

The current time is **05:43** in Tunisia and **06:43** in France.


In [8]:
import io
import json

data = {"important_data": [1, 10, -1541]}
binary_io = io.BytesIO(json.dumps(data).encode("ascii"))

upload_metadata = toolPythonSession.upload_file(
    data=binary_io, remote_file_path="important_data.json"
)

code = f"""
import json

with open("{upload_metadata.full_path}") as f:
    data = json.load(f)

sum(data['important_data'])
"""

toolPythonSession.execute(code)

{'$id': '2',
 'status': 'Success',
 'stdout': '',
 'stderr': '',
 'result': -1530,
 'executionTimeInMilliseconds': 6}

In [ ]:
# from langchain_azure_dynamic_sessions import SessionsBashTool

# tool = SessionsBashTool(pool_management_endpoint=sessionpool_management_endpoint, access_token_provider=access_token_provider)

# response = tool.execute("echo Hello from the session!")

# # response

# # # upload a file to the session
# # tool.upload_file(local_file_path="./rg.tf", remote_file_path="/mnt/user/rg.tf")

# # # list files in the session
# # files = tool.list_files()

# # # download a file from the session
# # tool.download_file(remote_file_path="/mnt/user/output.txt", local_file_path="./output.txt")